Restarted .venv (Python 3.10.16)

In [1]:
from sentence_transformers import SentenceTransformer, models
import torch

word_embedding_model = models.Transformer(
    'emilyalsentzer/Bio_ClinicalBERT',
    max_seq_length=512,
    model_args={"torch_dtype": torch.float32} 
)

pooling_model = models.Pooling(
    word_embedding_model.get_word_embedding_dimension(),
    pooling_mode_cls_token=False,
    pooling_mode_mean_tokens=True,
    pooling_mode_max_tokens=False
)

model = SentenceTransformer(modules=[word_embedding_model, pooling_model], device='mps')

In [2]:
import os
os.getcwd()
os.chdir('../..')
os.getcwd()

'/Users/nkapila6/Code/nlpvise/src'

In [3]:
model

SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
)

In [4]:
import pandas as pd

In [5]:
df = pd.read_pickle('data/preprocess_clean_remove_rare.pkl')

In [6]:
df.head(2)

,Unnamed: 0,ROW_ID,SUBJECT_ID,HADM_ID,CHARTDATE,CHARTTIME,STORETIME,CATEGORY,DESCRIPTION,CGID,ISERROR,TEXT,PTEXT,FTEXT
0,0,316398,31608,152365.0,2133-01-19,2133-01-19 06:53:00,2133-01-19 06:53:16,Nursing,Nursing Progress Note,19650.0,NaN,"TITLE:\n Respiratory failure, acute (not ARD...","[titl, respiratori, failur, acut, doctor, last...","[titl, respiratori, failur, acut, doctor, last..."
1,1,315910,31608,152365.0,2133-01-13,2133-01-13 06:21:00,2133-01-13 06:21:26,Nursing,Nursing Progress Note,21198.0,NaN,Ineffective Coping\n Assessment:\n Pt is n...,"[ineffect, cope, assess, pt, dnr, pt, ask, res...","[ineffect, cope, assess, pt, dnr, pt, ask, res..."


In [7]:
df['SENT'] = df['FTEXT'].apply(lambda tokens: ' '.join(tokens))

In [8]:
df.head(2)

,Unnamed: 0,ROW_ID,SUBJECT_ID,HADM_ID,CHARTDATE,CHARTTIME,STORETIME,CATEGORY,DESCRIPTION,CGID,ISERROR,TEXT,PTEXT,FTEXT,SENT
0,0,316398,31608,152365.0,2133-01-19,2133-01-19 06:53:00,2133-01-19 06:53:16,Nursing,Nursing Progress Note,19650.0,NaN,"TITLE:\n Respiratory failure, acute (not ARD...","[titl, respiratori, failur, acut, doctor, last...","[titl, respiratori, failur, acut, doctor, last...",titl respiratori failur acut doctor last name ...
1,1,315910,31608,152365.0,2133-01-13,2133-01-13 06:21:00,2133-01-13 06:21:26,Nursing,Nursing Progress Note,21198.0,NaN,Ineffective Coping\n Assessment:\n Pt is n...,"[ineffect, cope, assess, pt, dnr, pt, ask, res...","[ineffect, cope, assess, pt, dnr, pt, ask, res...",ineffect cope assess pt dnr pt ask resp tech c...


In [9]:
df['SENT'][5]

'respiratori failur acut doctor last name assess action respons plan ineffect cope assess action respons plan'

In [10]:
sentences = df['SENT'].tolist()

In [11]:
df.shape

(142110, 15)

In [12]:
len(sentences)

142110

In [13]:
embeddings = model.encode(sentences, batch_size=16, show_progress_bar=True)

Batches:   0%|          | 0/8882 [00:00<?, ?it/s]

In [20]:
import pickle

with open("data/embeddings_MEAN.pkl", "wb") as f:
    pickle.dump(embeddings, f)

In [15]:
type(embeddings)

numpy.ndarray

In [16]:
embeddings

array([[ 0.20631364, -0.44404224, -0.12459257, ...,  0.07726249,
        -0.28615013, -0.00592945],
       [ 0.23262683, -0.6760833 , -0.05741224, ...,  0.05930402,
        -0.17492321,  0.05269586],
       [ 0.19450292, -0.12628329, -0.28447592, ...,  0.08679035,
        -0.27400795, -0.21470568],
       ...,
       [ 0.19077775, -0.15893398, -0.01699762, ...,  0.2521493 ,
        -0.19124061, -0.01828537],
       [ 0.25270894, -0.41641697, -0.10034665, ...,  0.14024025,
        -0.1844983 ,  0.10219409],
       [ 0.19960865, -0.33914346, -0.23394534, ...,  0.18734786,
        -0.26054907,  0.10926247]], dtype=float32)

In [17]:
df['EMBEDDING'] = list(embeddings)

In [18]:
df.head()

,Unnamed: 0,ROW_ID,SUBJECT_ID,HADM_ID,CHARTDATE,CHARTTIME,STORETIME,CATEGORY,DESCRIPTION,CGID,ISERROR,TEXT,PTEXT,FTEXT,SENT,EMBEDDING
0,0,316398,31608,152365.0,2133-01-19,2133-01-19 06:53:00,2133-01-19 06:53:16,Nursing,Nursing Progress Note,19650.0,NaN,"TITLE:\n Respiratory failure, acute (not ARD...","[titl, respiratori, failur, acut, doctor, last...","[titl, respiratori, failur, acut, doctor, last...",titl respiratori failur acut doctor last name ...,"[0.20631364, -0.44404224, -0.12459257, 0.24738..."
1,1,315910,31608,152365.0,2133-01-13,2133-01-13 06:21:00,2133-01-13 06:21:26,Nursing,Nursing Progress Note,21198.0,NaN,Ineffective Coping\n Assessment:\n Pt is n...,"[ineffect, cope, assess, pt, dnr, pt, ask, res...","[ineffect, cope, assess, pt, dnr, pt, ask, res...",ineffect cope assess pt dnr pt ask resp tech c...,"[0.23262683, -0.6760833, -0.057412244, 0.35846..."
2,2,316202,31608,152365.0,2133-01-16,2133-01-16 03:53:00,2133-01-16 03:53:57,Nursing,Nursing Progress Note,14442.0,NaN,"Respiratory failure, acute (not ARDS/[**Doctor...","[respiratori, failur, acut, doctor, last, name...","[respiratori, failur, acut, doctor, last, name...",respiratori failur acut doctor last name asses...,"[0.19450292, -0.12628329, -0.28447592, 0.27614..."
3,3,316288,31608,152365.0,2133-01-17,2133-01-17 15:10:00,2133-01-17 15:10:44,Nursing,Nursing Progress Note,15065.0,NaN,"Respiratory failure, acute (not ARDS/[**Doctor...","[respiratori, failur, acut, doctor, last, name...","[respiratori, failur, acut, doctor, last, name...",respiratori failur acut doctor last name asses...,"[0.15812537, -0.5567824, -0.12215112, 0.418693..."
4,5,316296,31608,152365.0,2133-01-17,2133-01-17 15:10:00,2133-01-17 18:08:06,Nursing,Nursing Progress Note,15065.0,NaN,"Respiratory failure, acute (not ARDS/[**Doctor...","[respiratori, failur, acut, doctor, last, name...","[respiratori, failur, acut, doctor, last, name...",respiratori failur acut doctor last name asses...,"[0.15812537, -0.5567824, -0.12215112, 0.418693..."


In [19]:
df.to_pickle('data/notes_with_embeddings_MEAN.pkl')